# Forest account analysis

## Purpose and scope

This notebook inspects the current forest-account calculation from the committed rate and area outputs. It follows the paper's selected quantity: annual carbon additions to living above- and below-ground forest biomass, by four forest biomes. The computations shown here are checks and summaries of the canonical outputs; they do not rebuild the 272 Gibbs/GFW raster tiles.

## How to use the notebook

Run from top to bottom. The notebook locates the repository root, loads the four-biome rate table and the paper-facing summary table, recomputes the area-times-rate identity, and displays the Pan comparison and pixel-dispersion sensitivity. The source data and main processing functions remain documented in `docs/DATA_SOURCES.md`, `docs/METHODS.md`, and `code/src/gcw/`.

## Units and reading the calculation

The rate is in tonnes of carbon per hectare per year. Multiplying by forest area in hectares gives tonnes of carbon per year. Multiplication by 44/12 converts carbon mass to CO₂ mass; division by 10¹² expresses the result in GtCO₂ per year. The spatial standard deviation describes variation among source pixels and is displayed separately; it is not a confidence interval for the biome mean.

## 1. Set up paths and load the canonical outputs

This cell only locates already prepared files. If a required file is absent, the error identifies the missing path rather than silently substituting an older vintage.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
if not (ROOT / 'pyproject.toml').is_file():
    raise FileNotFoundError('Run this notebook from within the repository.')

DATA = ROOT / 'data' / 'derived'
RESULTS = ROOT / 'results'
TABLES = RESULTS / 'tables' / 'manuscript'
VALIDATION = RESULTS / 'validation'
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
print(f'Repository: {ROOT}')

Repository: C:\Users\aurel\Desktop\M2\Stage ecoseas-csm\CSM\BNC2 paper\repo


## 2. Inspect biome rates and paper-facing physical totals

`gfw_gibbs_biome_rates_non_mangrove.csv` contains the rate numerator/denominator summaries after mangrove source pixels have been excluded. `Supplementary_forest_biome_accounting.csv` combines these rates with the forest areas allocated to the four biomes. `TableS1_forest_biome_additions.csv` is the compact table intended for the paper supplement.

In [2]:
rates = pd.read_csv(DATA / 'gfw_gibbs_biome_rates_non_mangrove.csv')
accounting = pd.read_csv(TABLES / 'Supplementary_forest_biome_accounting.csv')
table_s1 = pd.read_csv(TABLES / 'TableS1_forest_biome_additions.csv')

rate_columns = ['biome', 'rate_tc_ha_yr_mean', 'rate_tc_ha_yr_spatial_sd', 'area_ha', 'pixel_count', 'excluded_mangrove_pixel_count', 'period', 'pool']
display(rates[[c for c in rate_columns if c in rates.columns]])
display(accounting)
display(table_s1)

,biome,rate_tc_ha_yr_mean,rate_tc_ha_yr_spatial_sd,area_ha,pixel_count,excluded_mangrove_pixel_count,period,pool
0,tropical,0.934399,0.926969,1.997984e+09,26376315835,137822459,2001-2023 mean annual rate,above- and below-ground living biomass
1,subtropical,1.370738,1.322735,3.380006e+08,5152437472,1995474,2001-2023 mean annual rate,above- and below-ground living biomass
2,temperate,1.430069,0.801324,6.040931e+08,11725272478,29570,2001-2023 mean annual rate,above- and below-ground living biomass
3,boreal,0.618633,0.261215,1.101383e+09,28192042781,0,2001-2023 mean annual rate,above- and below-ground living biomass


,biome,area_ha,rate_tc_ha_yr_mean,rate_tc_ha_yr_spatial_sd,period,source,gross_additions_tc_yr,gross_additions_gtco2_yr,spatial_rate_low_tc_ha_yr,spatial_rate_high_tc_ha_yr,spatial_rate_interpretation
0,tropical,1.852745e+09,0.934399,0.926969,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",1.731202e+09,6.347741,0.007429,1.861368,Spatial dispersion sensitivity; not a confiden...
1,subtropical,4.363417e+08,1.370738,1.322735,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",5.981103e+08,2.193071,0.048003,2.693473,Spatial dispersion sensitivity; not a confiden...
2,temperate,6.833128e+08,1.430069,0.801324,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",9.771843e+08,3.583009,0.628744,2.231393,Spatial dispersion sensitivity; not a confiden...
3,boreal,1.163864e+09,0.618633,0.261215,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",7.200044e+08,2.640016,0.357418,0.879848,Spatial dispersion sensitivity; not a confiden...
4,Total,4.136263e+09,NaN,NaN,2001-2023 source rates; FRA 2025 forest area,"Gibbs et al. (2025), Gibbs/GFW; FRA 2025 and G...",4.026501e+09,14.763837,NaN,NaN,Spatial dispersion sensitivity; not a confiden...


,Forest biome,Forest area (Mha),Removal rate (tC ha⁻¹ yr⁻¹),Gross additions (GtCO₂ yr⁻¹),Global share (%)
0,Tropical,"1,852.7",0.934,6.348,43.0
1,Subtropical,436.3,1.371,2.193,14.9
2,Temperate,683.3,1.430,3.583,24.3
3,Boreal,"1,163.9",0.619,2.640,17.9
4,Total,"4,136.2",—,14.764,100.0


## 3. Recalculate the area-times-rate identity

For each biome, the check below recalculates the physical flow from the area and rate columns. It compares this result with the stored output and then reports the rounded global total. A small floating-point residual is expected because the stored output may retain more decimal places than the display table.

In [3]:
biomes = accounting.loc[accounting['biome'].str.lower().ne('total')].copy()
biomes['recalculated_gtco2_yr'] = (
    biomes['area_ha'] * biomes['rate_tc_ha_yr_mean'] * (44 / 12) / 1e12
)
biomes['difference_gtco2_yr'] = biomes['recalculated_gtco2_yr'] - biomes['gross_additions_gtco2_yr']
display(biomes[['biome', 'area_ha', 'rate_tc_ha_yr_mean', 'recalculated_gtco2_yr', 'gross_additions_gtco2_yr', 'difference_gtco2_yr']])
print('Recalculated total (GtCO2/year):', biomes['recalculated_gtco2_yr'].sum())
print('Stored total (GtCO2/year):', biomes['gross_additions_gtco2_yr'].sum())

,biome,area_ha,rate_tc_ha_yr_mean,recalculated_gtco2_yr,gross_additions_gtco2_yr,difference_gtco2_yr
0,tropical,1.852745e+09,0.934399,0.006348,6.347741,-6.341393
1,subtropical,4.363417e+08,1.370738,0.002193,2.193071,-2.190878
2,temperate,6.833128e+08,1.430069,0.003583,3.583009,-3.579426
3,boreal,1.163864e+09,0.618633,0.002640,2.640016,-2.637376


Recalculated total (GtCO2/year): 0.014763837139138728
Stored total (GtCO2/year): 14.763837139138728


## 4. Keep pixel dispersion separate from rate uncertainty

The source-pixel standard deviation is shown beside each biome mean to make spatial heterogeneity visible. It is not an estimated sampling standard error and should not be read as a confidence interval. The accounting uncertainty draws and this spatial-dispersion sensitivity answer different questions, so they stay in separate columns and outputs.

In [4]:
display(rates[['biome', 'rate_tc_ha_yr_mean', 'rate_tc_ha_yr_spatial_sd', 'spatial_sd_interpretation']])

,biome,rate_tc_ha_yr_mean,rate_tc_ha_yr_spatial_sd,spatial_sd_interpretation
0,tropical,0.934399,0.926969,spatial dispersion; not an estimate of rate un...
1,subtropical,1.370738,1.322735,spatial dispersion; not an estimate of rate un...
2,temperate,1.430069,0.801324,spatial dispersion; not an estimate of rate un...
3,boreal,0.618633,0.261215,spatial dispersion; not an estimate of rate un...


## 5. Compare with the published Pan rates

Pan is retained as the paper's rate comparator. This table places the published values alongside the current Gibbs/GFW estimates where the biome mapping permits comparison. The comparison is descriptive; it does not replace the selected rate source.

In [5]:
pan = pd.read_csv(TABLES / 'Supplementary_Pan_rate_comparison.csv')
display(pan)

,biome,gibbs_gfw_rate_tc_ha_yr,gibbs_gfw_period,gibbs_gfw_source,pan_row,pan_rate_tc_ha_yr,pan_rate_sd_tc_ha_yr,pan_source,pan_period,comparison_note
0,boreal,0.618633,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",Boreal forest,0.278806,0.034851,Pan et al. (2024) Table 1 (2010-2019) / mean P...,2010-2019,Contextual comparison only: periods and source...
1,subtropical,1.370738,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",NaN,NaN,NaN,NaN,NaN,Contextual comparison only: periods and source...
2,temperate,1.430069,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",Temperate forest,0.865966,0.063674,Pan et al. (2024) Table 1 (2010-2019) / mean P...,2010-2019,Contextual comparison only: periods and source...
3,tropical,0.934399,2001-2023 mean annual rate,"Gibbs et al. (2025), Harvard Dataverse DOI 10....",All tropical forests,1.319406,0.214665,Pan et al. (2024) Table 1 (2010-2019) / mean P...,2010-2019,Contextual comparison only: periods and source...


## 6. Audit checklist

Review the displayed table for the four biome labels, units, rounded global total, and source period. The final cell reports whether the principal physical identity reconciles to numerical precision. It does not test the ecological assumptions behind the source rates or the FRA biome allocation.

In [6]:
assert set(biomes['biome'].str.lower()) == {'tropical', 'subtropical', 'temperate', 'boreal'}
max_abs_difference = biomes['difference_gtco2_yr'].abs().max()
print({'four_biomes_present': True, 'max_abs_flow_difference_gtco2_yr': float(max_abs_difference)})

{'four_biomes_present': True, 'max_abs_flow_difference_gtco2_yr': 6.341393263585186}
